# Multi-Host Distributed PyTorch Training with TorchTPU on GKE

This notebook demonstrates how to launch a **multi-host distributed PyTorch training job** across a multi-host Cloud TPU slice (e.g. 2 hosts × 4 chips = 8 TPU v5e chips) using **Kubeflow Trainer (`TrainJob`)**, **TorchTPU (`torch_tpu`)**, and **`torchrun`** on Google Kubernetes Engine (GKE).

---

## How Multi-Host TorchTPU Works on GKE

In a multi-host TPU slice (such as a `2x4` TPU v5e slice), multiple Kubernetes Pods are scheduled across distinct TPU host VMs connected by Google's high-bandwidth Inter-Chip Interconnect (ICI).

```mermaid
flowchart TD
    subgraph Notebook["Interactive Workspace (CPU or Single TPU)"]
      NB["JupyterLab / VS Code<br/>Submit TrainJob via Kubeflow SDK"]
    end

    subgraph TrainerControl["Kubeflow Trainer & GKE"]
      TJ["TrainJob (JobSet)"]
      GCW["GKE Common Webhooks (GCW)<br/>Injects TPU_WORKER_HOSTNAMES"]
    end

    subgraph Node0["TPU Host 0 (trainer-node-0-0)"]
      EVAL0["eval $(python3 -m torch.tpu.distributed.environment)<br/>Derives TORCH_TPU_SLICEBUILDER_ADDRESSES"]
      TRUN0["torchrun --nnodes=2 --node_rank=0 --nproc_per_node=4"]
      R0["Rank 0 (TPU Core 0)"]
      R1["Rank 1 (TPU Core 1)"]
      R2["Rank 2 (TPU Core 2)"]
      R3["Rank 3 (TPU Core 3)"]
    end

    subgraph Node1["TPU Host 1 (trainer-node-0-1)"]
      EVAL1["eval $(python3 -m torch.tpu.distributed.environment)<br/>Derives TORCH_TPU_SLICEBUILDER_ADDRESSES"]
      TRUN1["torchrun --nnodes=2 --node_rank=1 --nproc_per_node=4"]
      R4["Rank 4 (TPU Core 4)"]
      R5["Rank 5 (TPU Core 5)"]
      R6["Rank 6 (TPU Core 6)"]
      R7["Rank 7 (TPU Core 7)"]
    end

    NB -->|"client.train(...)"| TJ
    TJ --> GCW
    GCW --> Node0
    GCW --> Node1
    EVAL0 --> TRUN0 --> R0 & R1 & R2 & R3
    EVAL1 --> TRUN1 --> R4 & R5 & R6 & R7
    R0 <== "ICI Network (tpu_dist AllReduce)" ==> R4
    R1 <== "ICI Network (tpu_dist AllReduce)" ==> R5
    R2 <== "ICI Network (tpu_dist AllReduce)" ==> R6
    R3 <== "ICI Network (tpu_dist AllReduce)" ==> R7
```

### The Environment Bootstrap Contract

To coordinate across multi-host TPU slices, TorchTPU's C++ runtime requires specific distributed environment variables:
1. **`TPU_WORKER_HOSTNAMES`**: Automatically injected by **GKE Common Webhooks (GCW)** for Indexed Jobs requesting TPUs.
2. **`TORCH_TPU_SLICEBUILDER_ADDRESSES`**: Required by `torch_tpu` to establish SliceBuilder mesh endpoints for every worker rank across all hosts.
3. **`TORCH_TPU_TOPOLOGY`**: Physical TPU slice geometry (e.g., `2,4` or `2,2,1`).
4. **PyTorch Rendezvous (`NNODES`, `NODE_RANK`, `MASTER_ADDR`, `MASTER_PORT`)**: Needed by `torchrun` to synchronize processes.

> [!IMPORTANT]
> While GKE injects `TPU_WORKER_HOSTNAMES`, it does not set `TORCH_TPU_SLICEBUILDER_ADDRESSES` directly. TorchTPU provides a built-in launcher bootstrap module: `python3 -m torch.tpu.distributed.environment`. Running `eval "$(python3 -m torch.tpu.distributed.environment 2>/dev/null | grep -E '^[A-Z_]+=' | sed 's/^/export /')"` in each container before `torchrun` inspects the injected `TPU_WORKER_HOSTNAMES` and dynamically computes `TORCH_TPU_SLICEBUILDER_ADDRESSES`, `TORCH_TPU_TOPOLOGY`, `NODE_RANK`, and rendezvous addresses!

## 1. Setup & Environment Verification

Discover the current Kubernetes namespace and verify access to the cluster.

In [ ]:
import os
import pathlib
import sys
import time
from kubernetes import client as k8s_client, config as k8s_config

# Load in-cluster or local kubeconfig
try:
    k8s_config.load_incluster_config()
except Exception:
    k8s_config.load_kube_config()

def get_current_namespace() -> str:
    if os.environ.get("TENANT_NAMESPACE"):
        return os.environ["TENANT_NAMESPACE"]
    ns_path = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"
    if os.path.exists(ns_path):
        with open(ns_path) as f:
            return f.read().strip()
    return "default"

namespace = get_current_namespace()
print(f"Target Kubernetes namespace: {namespace}")

## 2. Define Modular Training Library and Entrypoint (`src/`)

In realistic machine learning projects, code is organized across multiple library modules rather than combined into a single monolithic script:
- **`src/models.py`**: Neural network architecture definitions (`MLPClassifier` using PyTorch `nn.Module`).
- **`src/dataset.py`**: Synthetic data generation, preprocessing, and distributed `DataLoader` / `DistributedSampler` helpers.
- **`src/train.py`**: Distributed training runner that imports `models` and `dataset`, configures the `torch_tpu` backend, wraps the model in PyTorch DDP (`DistributedDataParallel`), and executes the multi-epoch training loop.

Below, we create these modular files in a local `src/` directory.

In [ ]:
import pathlib

src_dir = pathlib.Path("src")
src_dir.mkdir(exist_ok=True)

# 1. Model architecture module (src/models.py)
models_code = '''"""Model architecture definitions."""
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLPClassifier(nn.Module):
    """Multi-layer perceptron classifier for image classification."""
    def __init__(self, in_features: int = 784, hidden1: int = 256, hidden2: int = 128, num_classes: int = 10):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)
'''

# 2. Dataset and data loader module (src/dataset.py)
dataset_code = '''"""Synthetic dataset generation and distributed DataLoader helpers."""
from typing import Tuple
import numpy as np
import torch
from torch.utils.data import DataLoader, DistributedSampler, TensorDataset

def generate_synthetic_data(
    num_samples: int = 16384,
    num_features: int = 784,
    num_classes: int = 10,
    seed: int = 42,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Generates synthetic clustered data for classification benchmarks."""
    np.random.seed(seed)
    prototypes = np.random.randn(num_classes, num_features).astype(np.float32)
    labels = np.random.randint(0, num_classes, size=(num_samples,))
    noise = np.random.randn(num_samples, num_features).astype(np.float32) * 0.5
    images = prototypes[labels] + noise
    return torch.tensor(images, dtype=torch.float32), torch.tensor(labels, dtype=torch.long)

def get_distributed_dataloader(
    batch_size: int = 128,
    num_samples: int = 16384,
    world_size: int = 1,
    rank: int = 0,
) -> Tuple[DataLoader, DistributedSampler]:
    """Builds a sharded DataLoader using DistributedSampler for DDP training."""
    images, labels = generate_synthetic_data(num_samples=num_samples, seed=42)
    dataset = TensorDataset(images, labels)
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    loader = DataLoader(dataset, batch_size=batch_size, sampler=sampler)
    return loader, sampler
'''

# 3. Main distributed training entrypoint (src/train.py)
train_code = '''#!/usr/bin/env python3
"""Multi-host distributed PyTorch DDP training entrypoint for TorchTPU."""
import os
import time
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.optim as optim
from torch.nn.parallel import DistributedDataParallel as DDP

# Import modular libraries mounted via Kubernetes ConfigMap
from dataset import get_distributed_dataloader
from models import MLPClassifier

# Ensure torch_tpu backend is registered
try:
    import torch_tpu
except ImportError:
    pass

def main():
    # 1. Initialize process group with tpu_dist backend
    if not dist.is_initialized():
        dist.init_process_group(backend="tpu_dist")
    device = torch.device("tpu")

    rank = dist.get_rank()
    world_size = dist.get_world_size()
    host_id = os.environ.get("TPU_WORKER_ID", os.environ.get("NODE_RANK", "0"))
    local_rank = os.environ.get("LOCAL_RANK", "0")

    print(
        f"[Host {host_id} | LocalRank {local_rank} | GlobalRank {rank}/{world_size}] "
        f"Initialized TorchTPU DDP worker on device: {device}",
        flush=True,
    )

    # 2. Synchronize random seeds for model weights across all ranks
    torch.manual_seed(42)

    # 3. Model setup on TPU wrapped in DDP
    model = MLPClassifier().to(device=device, dtype=torch.bfloat16)
    ddp_model = DDP(model)

    # 4. Distributed DataLoader sharded across all ranks
    loader, sampler = get_distributed_dataloader(
        batch_size=128,
        num_samples=16384,
        world_size=world_size,
        rank=rank,
    )

    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.SGD(ddp_model.parameters(), lr=0.05)

    epochs = 5
    total_start = time.time()

    for epoch in range(1, epochs + 1):
        sampler.set_epoch(epoch)
        epoch_start = time.time()
        running_loss = 0.0
        total_samples = 0

        for batch_images, batch_labels in loader:
            batch_images = batch_images.to(device=device, dtype=torch.bfloat16)
            batch_labels = batch_labels.to(device=device)

            optimizer.zero_grad()
            outputs = ddp_model(batch_images)
            loss = loss_fn(outputs, batch_labels)
            loss.backward()
            optimizer.step()

            # Execute .item() synchronously across all ranks to prevent divergence
            current_loss = loss.detach().item()
            running_loss += current_loss * batch_images.size(0)
            total_samples += batch_images.size(0)

        epoch_loss = running_loss / total_samples
        duration = time.time() - epoch_start
        throughput = (total_samples * world_size) / duration

        if rank == 0:
            print(
                f"Epoch {epoch:2d}/{epochs:2d} | Avg Loss: {epoch_loss:.4f} | "
                f"Throughput: {throughput:.0f} samples/s ({duration:.2f}s)",
                flush=True,
            )

    if rank == 0:
        print(
            f"\\n[Rank 0] Multi-host distributed training successfully finished "
            f"in {time.time() - total_start:.2f}s across {world_size} TPU cores!",
            flush=True,
        )

    dist.destroy_process_group()

if __name__ == "__main__":
    main()
'''

(src_dir / "models.py").write_text(models_code)
(src_dir / "dataset.py").write_text(dataset_code)
(src_dir / "train.py").write_text(train_code)

print("Modular training code written to src/:")
for f in sorted(src_dir.glob("*.py")):
    print(f" - {f.name} ({f.stat().st_size} bytes)")

## 3. Submit Multi-Host Distributed Training (`submit_multihost_training`)

All Kubernetes plumbing (ConfigMap packaging, volume mounts, `RuntimePatch`, nodeSelectors, tolerations, and TorchTPU bootstrapping) is encapsulated inside `submit_multihost_training()`.

Users only need to provide high-level training parameters:
- **`src_dir`**: Directory containing your modular library files (`"src"`).
- **`main_script`**: Training entrypoint script (`"train.py"`).
- **`num_nodes`**: Number of TPU host nodes in the slice (`2`).
- **`tpus_per_node`**: Physical TPU chips per host (`4`).
- **`compute_class`**: GKE TPU ComputeClass (`"tpu-v5-8-multi-host"`).
- **`torch_tpu_topology`**: Full slice topology (`"2,4,1"`).

*(To inspect the underlying Kubernetes implementation and SDK options, see [`tpu_trainer.py`](tpu_trainer.py).)*

In [ ]:
from tpu_trainer import submit_multihost_training

# Submit the distributed training job with a single Python call
job_name = submit_multihost_training(
    src_dir="src",
    main_script="train.py",
    num_nodes=2,
    tpus_per_node=4,
    compute_class="tpu-v5-8-multi-host",
    torch_tpu_topology="2,4,1",
)

## 4. Monitor TrainJob & Active Host Pods

Watch the `TrainJob` and inspect the worker Pods scheduled across the TPU nodes. Once all host pods transition to `Running`, distributed training begins.

In [ ]:
from tpu_trainer import wait_for_job_pods

# Wait until all host pods are active and Running
wait_for_job_pods(job_name, num_nodes=2)

## 5. Stream Distributed Logs Across TPU Hosts

Stream output from all active worker pods. Observe how each host initializes its local TorchTPU environment, exchanges SliceBuilder addresses, launches its 4 local TPU processes via `torchrun`, and executes distributed training across all 8 cores.

In [ ]:
from tpu_trainer import stream_job_logs

# Stream and filter distributed training logs across all TPU worker nodes
stream_job_logs(job_name, num_nodes=2)

## 6. Cleanup Resources

When you have finished inspecting the distributed training logs, execute this cell to delete the `TrainJob` and the associated code `ConfigMap`.

In [ ]:
from tpu_trainer import delete_training_job

# Clean up TrainJob and the uploaded code ConfigMap
delete_training_job(job_name)